# STAT 301 Project: Predicting Airbnb Prices in European Cities

Group 16:

In [1]:
# Package Loading
library(tidyverse)
library(ggplot2)
library(ggmap)
library(patchwork)
library(janitor)
library(broom)
library(car)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.1     ✔ tibble    3.2.1
✔ lubridate 1.9.3     ✔ tidyr     1.3.1
✔ purrr     1.0.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


ERROR: Error in library(ggmap): there is no package called ‘ggmap’


## 1. Introduction

In this report we will...

In section 2 we perform EDA, ..., section 3 contains a discussion of our ...

## 2. Methods & Results

### 2.1 Exploratory Data Analysis

In [2]:
# Main developer: Nathan Zhang
# Loading the data and select the relevant columns

airbnb_data <- read_csv("https://raw.githubusercontent.com/NathanPalaiologos/STAT-301-Project/main/data/airbnb_europe_data.csv") |>
    filter(city %in% c("budapest", "london", "rome")) |>
    glimpse()


Rows: 51707 Columns: 21
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (2): room_type, city
dbl (15): realSum, person_capacity, multi, biz, cleanliness_rating, guest_sa...
lgl  (4): room_shared, room_private, host_is_superhost, weekend

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


Rows: 23,042
Columns: 21
$ realSum                    <dbl> 238.99046, 300.79428, 162.38191, 118.43775,…
$ room_type                  <chr> "Entire home/apt", "Entire home/apt", "Enti…
$ room_shared                <lgl> FALSE, FALSE, FALSE, FALSE, FALSE, FALSE, F…
$ room_private               <lgl> FALSE, FALSE, FALSE, FALSE, FALSE, FALSE, F…
$ person_capacity            <dbl> 6, 6, 4, 2, 4, 4, 4, 4, 6, 4, 3, 4, 5, 6, 4…
$ host_is_superhost          <lgl> TRUE, FALSE, TRUE, FALSE, TRUE, FALSE, FALS…
$ multi                      <dbl> 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0…
$ biz                        <dbl> 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1…
$ cleanliness_rating         <dbl> 10, 9, 10, 9, 10, 9, 6, 9, 10, 10, 10, 10, …
$ guest_satisfaction_overall <dbl> 99, 98, 98, 92, 99, 91, 80, 92, 88, 96, 100…
$ bedrooms                   <dbl> 1, 2, 1, 1, 2, 2, 2, 1, 2, 1, 0, 1, 2, 2, 0…
$ dist                       <dbl> 0.3593550, 0.9294272, 2.4508403, 1.5594494,…
$ metro_dist   

#### Cleaning and Wrangling
* The variables `room_shared` and `room_private` have been removed because they provide redundant information that has already been captured by the variable `room_type`. Both `room_shared` and `room_private` are dummy variables that correspond to the categories of `room_type` (as also demonstrated in the code cell below):
|room_type|room_shared|room_private|
|:--------|:----------|:-----------|
|Entire home/apt|FALSE|FALSE|
|Private room|FALSE|TRUE|
|Shared room|TRUE|FALSE|

  This correspondence tells us that no new information is gained by keeping the dummy variables. Keeping only the `room_type` variable reduces redundancy and simplifies model interpretation.

* Furthermore, the variables `attr_index` and `rest_index` have been removed because they represent the unstandardized indices which can make them difficult to compare across listings. Instead, the variables `attr_index_norm` and `rest_index_norm` have been retained as they are the normalized versions that provide relative measures. These normalized values are scale-independent, allowing for better comparability across cities.

In [9]:
#Main developer: Myra Tyagi

#Checks how the three variables relate
airbnb_data %>%
  select(room_type, room_shared, room_private) %>%
  distinct()

clean_airbnb_data <- airbnb_data %>%

#Removing unnecessary columns
    select(-room_shared, -room_private, -attr_index, -rest_index) %>%

#Converting room_type and city to factors
    mutate(room_type = factor(room_type), city = factor(city)) %>%
    droplevels() %>%

#Renaming columns for clarity
    rename(multipleRooms = multi,
           business = biz) %>%
    glimpse()

#Checks for typos
unique(clean_airbnb_data$room_type)
unique(clean_airbnb_data$city)

#Checking for missing values
colSums(is.na(clean_airbnb_data))

room_type,room_shared,room_private
<chr>,<lgl>,<lgl>
Entire home/apt,FALSE,FALSE
Private room,FALSE,TRUE
Shared room,TRUE,FALSE


Rows: 23,042
Columns: 17
$ realSum                    <dbl> 238.99046, 300.79428, 162.38191, 118.43775,…
$ room_type                  <fct> Entire home/apt, Entire home/apt, Entire ho…
$ person_capacity            <dbl> 6, 6, 4, 2, 4, 4, 4, 4, 6, 4, 3, 4, 5, 6, 4…
$ host_is_superhost          <lgl> TRUE, FALSE, TRUE, FALSE, TRUE, FALSE, FALS…
$ multipleRooms              <dbl> 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0…
$ business                   <dbl> 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1…
$ cleanliness_rating         <dbl> 10, 9, 10, 9, 10, 9, 6, 9, 10, 10, 10, 10, …
$ guest_satisfaction_overall <dbl> 99, 98, 98, 92, 99, 91, 80, 92, 88, 96, 100…
$ bedrooms                   <dbl> 1, 2, 1, 1, 2, 2, 2, 1, 2, 1, 0, 1, 2, 2, 0…
$ dist                       <dbl> 0.3593550, 0.9294272, 2.4508403, 1.5594494,…
$ metro_dist                 <dbl> 0.3526430, 0.2002355, 0.2794518, 0.4779711,…
$ attr_index_norm            <dbl> 24.116552, 100.000000, 9.755551, 11.433155,…
$ rest_index_no

[1] Entire home/apt Private room    Shared room    
Levels: Entire home/apt Private room Shared room

[1] budapest london   rome    
Levels: budapest london rome

realSum                  room_type 
                         0                          0 
           person_capacity          host_is_superhost 
                         0                          0 
             multipleRooms                   business 
                         0                          0 
        cleanliness_rating guest_satisfaction_overall 
                         0                          0 
                  bedrooms                       dist 
                         0                          0 
                metro_dist            attr_index_norm 
                         0                          0 
           rest_index_norm                        lng 
                         0                          0 
                       lat                       city 
                         0                          0 
                   weekend 
                         0

### 2.2 Analysis Plan

### 2.3 Results

## 3. Discussion

## 3. References